# 04. Ventilation Raw (인공호흡기 원본 추출)

## 목적
코호트 환자의 인공호흡기 사용 데이터를 **1시간 단위**로 추출

## 입력
- `sepsis_cohort.csv`: 기본 코호트

## 출력
- `ventilation_raw.csv`: 시간별 인공호흡기 상태 (1 row = 1 patient × 1 hour)

## 전처리 범위
- [x] 데이터 추출 (procedureevents)
- [x] 시간 범위 확장 (start~end → 1시간 단위)
- [ ] 슬라이딩 윈도우 적용 → merge 단계에서

## Item ID
- 225792: Invasive Mechanical Ventilation

In [1]:
from pathlib import Path
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
import duckdb
import pandas as pd
import os

DB_PATH = RAW_DIR / "mimic_total.duckdb"
INPUT_DIR = PROCESSED_DIR
OUTPUT_DIR = PROCESSED_DIR

con = duckdb.connect(DB_PATH)
print("=== 04. Ventilation Raw 추출 시작 ===")

=== 04. Ventilation Raw 추출 시작 ===


## Step 1: 코호트 로드

In [2]:
print("Step 1: 코호트 로드")

cohort_path = INPUT_DIR / "sepsis_cohort.csv"
df_cohort = pd.read_csv(cohort_path, parse_dates=['intime', 'outtime'])

print(f"✓ 코호트 로드 완료: {len(df_cohort):,}명")

con.execute("CREATE OR REPLACE TEMP TABLE cohort AS SELECT * FROM read_csv_auto(?)", [str(cohort_path)])


Step 1: 코호트 로드
✓ 코호트 로드 완료: 18,001명


## Step 2: Ventilation 데이터 추출 (1시간 단위로 확장)

In [3]:
print("\nStep 2: Ventilation 데이터 추출")

vent_query = """
WITH vent_expanded AS (
    -- procedureevents의 starttime~endtime을 1시간 단위로 확장
    SELECT
        c.stay_id,
        UNNEST(generate_series(
            date_trunc('hour', CAST(pe.starttime AS TIMESTAMP)),
            date_trunc('hour', CAST(pe.endtime AS TIMESTAMP)),
            INTERVAL '1' HOUR
        )) AS charttime_h
    FROM cohort c
    INNER JOIN procedureevents pe ON c.stay_id = pe.stay_id
    WHERE pe.itemid = '225792'  -- Invasive Mechanical Ventilation
      AND pe.starttime IS NOT NULL
      AND pe.endtime IS NOT NULL
)
SELECT
    stay_id,
    charttime_h,
    1 as vent_flag  -- 해당 시간에 ventilation 사용 중
FROM vent_expanded
GROUP BY stay_id, charttime_h
ORDER BY stay_id, charttime_h
"""

print("  쿼리 실행 중...")
df_vent = con.execute(vent_query).df()

print(f"✓ Ventilation 추출 완료")
print(f"  - 총 행 수: {len(df_vent):,}개 (ventilation 사용 중인 시점만)")
print(f"  - 고유 환자: {df_vent['stay_id'].nunique():,}명")


Step 2: Ventilation 데이터 추출
  쿼리 실행 중...
✓ Ventilation 추출 완료
  - 총 행 수: 1,004,244개 (ventilation 사용 중인 시점만)
  - 고유 환자: 11,426명


## Step 3: 통계 확인

In [4]:
print("\nStep 3: 통계 확인")

# 환자당 ventilation 시간
vent_hours_per_patient = df_vent.groupby('stay_id').size()

print(f"\n=== Ventilation 통계 ===")
print(f"  - Ventilation 사용 환자: {df_vent['stay_id'].nunique():,}명")
print(f"  - 전체 코호트 대비: {df_vent['stay_id'].nunique() / len(df_cohort) * 100:.1f}%")
print(f"\n환자당 Ventilation 시간:")
print(f"  - 평균: {vent_hours_per_patient.mean():.1f}시간")
print(f"  - 중앙값: {vent_hours_per_patient.median():.1f}시간")
print(f"  - 최대: {vent_hours_per_patient.max():,}시간")


Step 3: 통계 확인

=== Ventilation 통계 ===
  - Ventilation 사용 환자: 11,426명
  - 전체 코호트 대비: 63.5%

환자당 Ventilation 시간:
  - 평균: 87.9시간
  - 중앙값: 29.0시간
  - 최대: 2,415시간


## Step 4: 저장

In [5]:
print("\n" + "="*60)
print("CSV 저장")
print("="*60)

output_path = OUTPUT_DIR / "ventilation_raw.csv"
df_vent.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: ventilation_raw.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_vent):,}개")
print(f"  - 경로: {output_path}")
print(f"\n참고: vent_flag=1인 행만 포함 (0은 merge 시 생성)")


CSV 저장

✓ 저장 완료: ventilation_raw.csv
  - 파일 크기: 29.69 MB
  - 행 수: 1,004,244개
  - 경로: DATA/processed/ventilation_raw.csv

참고: vent_flag=1인 행만 포함 (0은 merge 시 생성)


In [6]:
print("\n=== 샘플 데이터 ===")
df_vent.head(10)


=== 샘플 데이터 ===


,stay_id,charttime_h,vent_flag
0,30003749,2180-06-06 16:00:00,1
1,30003749,2180-06-06 17:00:00,1
2,30003749,2180-06-06 18:00:00,1
3,30003749,2180-06-06 19:00:00,1
4,30003749,2180-06-06 20:00:00,1
5,30003749,2180-06-06 21:00:00,1
6,30003749,2180-06-06 22:00:00,1
7,30003749,2180-06-06 23:00:00,1
8,30003749,2180-06-07 00:00:00,1
9,30003749,2180-06-07 01:00:00,1


In [7]:
con.close()
print("\n=== 04. Ventilation Raw 추출 완료 ===")


=== 04. Ventilation Raw 추출 완료 ===
